# YOLO CIFAR10: Fine-Tuned Teacher → Pruned Student → Logits Distillation (Classification)

This notebook runs a YOLOv8n-cls **classification** pruning + logits distillation flow on CIFAR10:
1. Load a **CIFAR10-fine-tuned** YOLOv8n-cls teacher (10-class head).
2. Build a `MaseGraph` from the teacher weights and run unstructured pruning to create the student.
3. Distill teacher logits into the pruned student using CIFAR10 mini-batches (images + labels).
4. Evaluate teacher, pruned-only (no KD) student, and distilled student on CIFAR10 validation set.

In [1]:
cls_model_checkpoint = "data/cifar10_yolov8n_cls/runs/yolov8n_cls_cifar10_finetune4/weights/best.pt"
cls_device = "cuda"
cifar_root = "./data"

image_size = 32
cifar_batch_size = 16
cifar_train_subset_size = 2048
cifar_val_subset_size = 512

cifar_prune_sparsity = 0.3
cifar_kd_alpha = 0.8
cifar_kd_temperature = 2.0
cifar_kd_steps = 200
lr = 1e-4
seed = 42

## Imports and setup

In [2]:
import copy
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo.yolov8 import MaseYoloClassificationModel, patch_yolo

from mase_kd.core.losses import DistillationLossConfig, hard_label_ce_loss, soft_logit_kl_loss
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

print(f"Repo root: {repo_root}")
print(f"Using src path: {src_path}")

/usr/local/lib/python3.11/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Repo root: /workspace/mase-kd
Using src path: /workspace/mase-kd/src


In [3]:
if cls_device == "cuda" and not torch.cuda.is_available():
    cls_device = "cpu"

torch.manual_seed(seed)
random.seed(seed)

print(f"Using device: {cls_device}")

Using device: cuda


## CIFAR10 image dataset and dataloaders

In [4]:
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

cifar_transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
cifar_transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_full = datasets.CIFAR10(root=cifar_root, train=True, transform=cifar_transform_train, download=True)
val_full = datasets.CIFAR10(root=cifar_root, train=False, transform=cifar_transform_eval, download=True)

subset_rng = torch.Generator().manual_seed(seed)
train_indices = torch.randperm(len(train_full), generator=subset_rng)[:cifar_train_subset_size]
val_indices = torch.randperm(len(val_full), generator=subset_rng)[:cifar_val_subset_size]

train_ds = Subset(train_full, train_indices.tolist())
val_ds = Subset(val_full, val_indices.tolist())

# drop_last=True keeps batch size fixed for the FX-specialized pruned student graph.
train_loader = DataLoader(
    train_ds,
    batch_size=cifar_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cifar_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"Train subset: {len(train_ds)} | Val subset: {len(val_ds)}")

Train subset: 2048 | Val subset: 512


## Load CIFAR10-fine-tuned teacher (YOLOv8n-cls, 10 classes)

In [5]:
ultra_teacher = YOLO(cls_model_checkpoint)
teacher_cls_model = ultra_teacher.model.to(cls_device)
teacher_cls_model.eval()

nc = ultra_teacher.model.yaml.get("nc", 10)
print(f"Loaded CIFAR10-fine-tuned teacher: {cls_model_checkpoint}")
print(f"Teacher num_classes (nc): {nc}")

Loaded CIFAR10-fine-tuned teacher: data/cifar10_yolov8n_cls/runs/yolov8n_cls_cifar10_finetune4/weights/best.pt
Teacher num_classes (nc): 10


## Build MASE graph from teacher weights and prune to create student

In [6]:
trace_input_cls = torch.randn(cifar_batch_size, 3, image_size, image_size)

# Build a MaseYoloClassificationModel with the CIFAR10 class count, then load
# the fine-tuned teacher weights. We bypass get_yolo_classification_model()
# because it expects the original "yolov8n-cls.pt" name and 1000-class arch.
student_seed_cls_model = MaseYoloClassificationModel(cfg="yolov8n-cls.yaml", nc=nc)
student_seed_cls_model = patch_yolo(student_seed_cls_model)
student_seed_cls_model.load_state_dict(ultra_teacher.model.state_dict())
student_seed_cls_model.train()

mg_cls = MaseGraph(student_seed_cls_model, cf_args={"x": trace_input_cls})
mg_cls, _ = passes.init_metadata_analysis_pass(mg_cls)

placeholder_names_cls = [
    node.name for node in mg_cls.fx_graph.nodes if node.op == "placeholder"
]
dummy_in_cls = {name: trace_input_cls for name in placeholder_names_cls}

mg_cls, _ = passes.add_common_metadata_analysis_pass(
    mg_cls,
    pass_args={
        "dummy_in": dummy_in_cls,
        "add_value": True,
    },
)

pruning_config_cls = {
    "weight": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
    "activation": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
}

mg_cls, _ = passes.prune_transform_pass(mg_cls, pass_args=pruning_config_cls)
student_cls_model = mg_cls.model.to(cls_device)

print(f"Pruning complete ({cifar_prune_sparsity*100:.0f}% sparsity); using pruned teacher as student.")

Overriding model.yaml nc=1000 with nc=10

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             


  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    343050  ultralytics.nn.modules.head.Classify         [256, 10]                     
YOLOv8n-cls summary: 56 layers, 1,451,098 parameters, 1,451,098 gradients, 3.4 GFLOPs
tensor([[[[-4.0005e-03]],

         [[-1.5646e-01]],

         [[-2.3093e-04]],

         ...,

         [[-6.5909e-07]],

         [[-2.4836e-09]],

         [[-2.4167e-02]]],


        [[[-1.5091e-02]],

         [[-1.1201e-02]],

         [[-2.1692e-04]],

         ...,

         [[-1.6136e-06]],

         [[-1.4263e-03]],

         [[-8.2

/usr/local/lib/python3.11/dist-packages/torch/fx/_symbolic_trace.py:906: UserWarning: Was not able to add assertion to guarantee correct input x to specialized function. It is up to the user to make sure that your inputs match the inputs you specialized the function with.
  warnings.warn(
INFO     Pruning module: model_0_conv
INFO     Pruning module: model_1_conv
INFO     Pruning module: model_2_cv1_conv
INFO     Pruning module: model_2_m_0_cv1_conv
INFO     Pruning module: model_2_m_0_cv2_conv
INFO     Pruning module: model_2_cv2_conv
INFO     Pruning module: model_3_conv
INFO     Pruning module: model_4_cv1_conv
INFO     Pruning module: model_4_m_0_cv1_conv
INFO     Pruning module: model_4_m_0_cv2_conv
INFO     Pruning module: model_4_m_1_cv1_conv
INFO     Pruning module: model_4_m_1_cv2_conv
INFO     Pruning module: model_4_cv2_conv
INFO     Pruning module: model_5_conv
INFO     Pruning module: model_6_cv1_conv
INFO     Pruning module: model_6_m_0_cv1_conv
INFO     Pruning module: m

Pruning complete (30% sparsity); using pruned teacher as student.


## Distill teacher into pruned student on CIFAR10 images + labels

In [7]:
kd_config_cls = DistillationLossConfig(
    alpha=cifar_kd_alpha,
    temperature=cifar_kd_temperature,
)
distiller_cls = YOLOLogitsDistiller(
    teacher=teacher_cls_model,
    student=student_cls_model,
    kd_config=kd_config_cls,
    device=cls_device,
)

# Snapshot the pruned student before KD training for baseline comparison.
pruned_no_kd_model = copy.deepcopy(student_cls_model).to(cls_device)
pruned_no_kd_model.eval()


GraphModule(
  (model): Module(
    (0): Module(
      (conv): ParametrizedConv2d(
        3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False
        (parametrizations): ModuleDict(
          (weight): ParametrizationList(
            (0): FakeSparseWeight()
          )
        )
      )
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): SiLU()
    )
    (1): Module(
      (conv): ParametrizedConv2d(
        16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False
        (parametrizations): ModuleDict(
          (weight): ParametrizationList(
            (0): FakeSparseWeight()
          )
        )
      )
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): Module(
      (cv1): Module(
        (conv): ParametrizedConv2d(
          32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False
          (parametrizations): ModuleDict(
            (weight):

In [8]:
kd_config_cls = DistillationLossConfig(
    alpha=cifar_kd_alpha,
    temperature=cifar_kd_temperature,
)
distiller_cls = YOLOLogitsDistiller(
    teacher=teacher_cls_model,
    student=student_cls_model,
    kd_config=kd_config_cls,
    device=cls_device,
)

# Snapshot the pruned student before KD training for baseline comparison.
pruned_no_kd_model = copy.deepcopy(student_cls_model).to(cls_device)
pruned_no_kd_model.eval()

optimizer = torch.optim.Adam(student_cls_model.parameters(), lr=lr)
loss_history = []

def classification_hard_loss(student_output, batch):
    targets = batch["targets"]
    logits = YOLOLogitsDistiller._extract_logits_with_batch(student_output, targets.shape[0])
    if logits is None:
        raise ValueError(
            "Could not find student logits matching target batch size for hard CE loss. "
            "Check trace_input_cls batch size vs DataLoader batch size."
        )
    max_label = int(targets.max().item())
    if logits.shape[1] <= max_label:
        raise ValueError(
            f"Student logits class dim ({logits.shape[1]}) <= max target ({max_label})."
        )
    return hard_label_ce_loss(logits, targets)

train_iter = iter(train_loader)
for step in range(1, cifar_kd_steps + 1):
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    batch = {
        "images": images.to(cls_device),
        "targets": labels.to(cls_device),
    }

    if step == 1:
        with torch.no_grad():
            sample_out = student_cls_model(batch["images"])
        sample_logits = YOLOLogitsDistiller._extract_logits_with_batch(sample_out, batch["targets"].shape[0])
        if sample_logits is None:
            print("Step 1 debug: could not extract batch-aligned logits from student output.")
        else:
            print(
                f"Step 1 debug: extracted logits shape={tuple(sample_logits.shape)} "
                f"for targets shape={tuple(batch['targets'].shape)}"
            )

    output = distiller_cls.train_step(
        batch=batch,
        optimizer=optimizer,
    )
    loss_history.append(output.total_loss)

    if step == 1 or step % 10 == 0 or step == cifar_kd_steps:
        print(
            f"Step {step:03d} | total={output.total_loss:.6f} | "
            f"hard={output.hard_loss:.6f} | soft={output.soft_loss:.6f}"
        )

Step 1 debug: extracted logits shape=(16, 10) for targets shape=(16,)
Step 001 | total=4.333826 | hard=4.240063 | soft=4.357266
Step 010 | total=3.530729 | hard=2.650338 | soft=3.750827
Step 020 | total=3.413271 | hard=2.735177 | soft=3.582794
Step 030 | total=3.120715 | hard=2.559788 | soft=3.260947
Step 040 | total=2.442909 | hard=2.503818 | soft=2.427682
Step 050 | total=3.184476 | hard=2.445798 | soft=3.369145
Step 060 | total=2.580430 | hard=2.327498 | soft=2.643663
Step 070 | total=2.685579 | hard=2.531746 | soft=2.724037
Step 080 | total=2.697805 | hard=2.350674 | soft=2.784588
Step 090 | total=2.779903 | hard=2.445672 | soft=2.863461
Step 100 | total=3.753821 | hard=2.771698 | soft=3.999351
Step 110 | total=2.721195 | hard=2.436641 | soft=2.792334
Step 120 | total=2.929757 | hard=2.474514 | soft=3.043567
Step 130 | total=2.861628 | hard=2.473976 | soft=2.958541
Step 140 | total=3.054852 | hard=2.469369 | soft=3.201222
Step 150 | total=2.572263 | hard=2.390841 | soft=2.617618
St

## Evaluation: teacher vs pruned-only vs distilled student (CIFAR10 validation)

In [ ]:
import time

def _align_logits_for_kd(student_logits, teacher_logits):
    width = min(student_logits.shape[1], teacher_logits.shape[1])
    if width == 0:
        return None, None
    return student_logits[:, :width], teacher_logits[:, :width]

@torch.no_grad()
def evaluate_model_on_cifar10_val(model, loader, device):
    """Evaluate a model on the full validation loader. Returns top-1 accuracy, CE loss, and timing."""
    model.eval()
    batches = 0
    samples = 0
    total_forward_ms = 0.0
    correct_top1 = 0
    total_ce_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        logits = YOLOLogitsDistiller._extract_logits_with_batch(outputs, images.shape[0])
        if logits is None or logits.numel() == 0:
            continue

        total_forward_ms += (t1 - t0) * 1000.0
        batches += 1
        samples += images.shape[0]

        max_label = int(labels.max().item())
        if logits.shape[1] > max_label:
            preds = logits.argmax(dim=1)
            correct_top1 += int((preds == labels).sum().item())
            total_ce_loss += hard_label_ce_loss(logits, labels).item()

    return {
        "batches": batches,
        "samples": samples,
        "avg_forward_ms_per_batch": total_forward_ms / max(batches, 1),
        "top1_acc": correct_top1 / max(samples, 1),
        "avg_ce_loss": total_ce_loss / max(batches, 1),
    }

@torch.no_grad()
def evaluate_kd_loss(teacher, student, loader, device, temperature=2.0):
    """Compute average KD (KL-div) loss between teacher and student on the full validation loader."""
    teacher.eval()
    student.eval()
    total_kd_loss = 0.0
    used_batches = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        teacher_outputs = teacher(images)
        student_outputs = student(images)

        teacher_logits = YOLOLogitsDistiller._extract_logits_with_batch(teacher_outputs, labels.shape[0])
        student_logits = YOLOLogitsDistiller._extract_logits_with_batch(student_outputs, labels.shape[0])
        if teacher_logits is None or student_logits is None:
            continue

        student_logits, teacher_logits = _align_logits_for_kd(student_logits, teacher_logits)
        if student_logits is None or teacher_logits is None:
            continue

        total_kd_loss += soft_logit_kl_loss(student_logits, teacher_logits, temperature).item()
        used_batches += 1

    return total_kd_loss / max(used_batches, 1), used_batches

# --- Training loss summary ---
if "loss_history" in globals() and len(loss_history) > 0:
    print(f"Recorded {len(loss_history)} KD training steps")
    print(f"  First loss: {loss_history[0]:.6f}")
    print(f"  Last loss:  {loss_history[-1]:.6f}")
else:
    print("No training loss history found in current kernel session.")

# --- Evaluate all three models on the CIFAR10 validation set ---
print("\n--- CIFAR10 Validation Results ---\n")

teacher_metrics = evaluate_model_on_cifar10_val(teacher_cls_model, val_loader, cls_device)
student_metrics = evaluate_model_on_cifar10_val(student_cls_model, val_loader, cls_device)

pruned_no_kd_metrics = None
if "pruned_no_kd_model" in globals():
    pruned_no_kd_metrics = evaluate_model_on_cifar10_val(pruned_no_kd_model, val_loader, cls_device)

# KD loss: teacher vs distilled student
val_kd_loss, kd_batches = evaluate_kd_loss(
    teacher_cls_model, student_cls_model, val_loader, cls_device,
    temperature=cifar_kd_temperature,
)

print(f"Teacher (fine-tuned on CIFAR10):")
print(
    f"  top1_acc={teacher_metrics['top1_acc'] * 100.0:.2f}% | "
    f"CE_loss={teacher_metrics['avg_ce_loss']:.4f} | "
    f"fwd_ms/batch={teacher_metrics['avg_forward_ms_per_batch']:.2f} | "
    f"samples={teacher_metrics['samples']}"
)

if pruned_no_kd_metrics is not None:
    print(f"\nPruned student (NO distillation, {cifar_prune_sparsity*100:.0f}% sparsity):")
    print(
        f"  top1_acc={pruned_no_kd_metrics['top1_acc'] * 100.0:.2f}% | "
        f"CE_loss={pruned_no_kd_metrics['avg_ce_loss']:.4f} | "
        f"fwd_ms/batch={pruned_no_kd_metrics['avg_forward_ms_per_batch']:.2f} | "
        f"samples={pruned_no_kd_metrics['samples']}"
    )

print(f"\nDistilled student (pruned + {cifar_kd_steps} KD steps, alpha={cifar_kd_alpha}, T={cifar_kd_temperature}):")
print(
    f"  top1_acc={student_metrics['top1_acc'] * 100.0:.2f}% | "
    f"CE_loss={student_metrics['avg_ce_loss']:.4f} | "
    f"fwd_ms/batch={student_metrics['avg_forward_ms_per_batch']:.2f} | "
    f"samples={student_metrics['samples']}"
)

print(f"\nValidation KD loss (teacher vs distilled student, T={cifar_kd_temperature}): "
      f"{val_kd_loss:.6f} over {kd_batches} batches")

# --- Summary table ---
print("\n--- Summary ---")
print(f"{'Model':<40} {'Top-1 Acc':>10} {'CE Loss':>10}")
print(f"{'─'*40} {'─'*10} {'─'*10}")
print(f"{'Teacher (fine-tuned)':<40} {teacher_metrics['top1_acc']*100:>9.2f}% {teacher_metrics['avg_ce_loss']:>10.4f}")
if pruned_no_kd_metrics is not None:
    print(f"{'Pruned student (no KD)':<40} {pruned_no_kd_metrics['top1_acc']*100:>9.2f}% {pruned_no_kd_metrics['avg_ce_loss']:>10.4f}")
print(f"{'Distilled student (pruned + KD)':<40} {student_metrics['top1_acc']*100:>9.2f}% {student_metrics['avg_ce_loss']:>10.4f}")

Recorded 200 KD training steps
  First loss: 4.333826
  Last loss:  2.087877

--- CIFAR10 Validation Results ---

Teacher (fine-tuned on CIFAR10):
  top1_acc=75.78% | CE_loss=1.7894 | fwd_ms/batch=9.00 | samples=512

Pruned student (NO distillation, 30% sparsity):
  top1_acc=10.16% | CE_loss=2.6388 | fwd_ms/batch=18.67 | samples=512

Distilled student (pruned + 200 KD steps, alpha=0.8, T=2.0):
  top1_acc=9.38% | CE_loss=2.3724 | fwd_ms/batch=22.76 | samples=512

Validation KD loss (teacher vs distilled student, T=2.0): 0.088982 over 32 batches

--- Summary ---
Model                                     Top-1 Acc    CE Loss
──────────────────────────────────────── ────────── ──────────
Teacher (fine-tuned)                         75.78%     1.7894
Pruned student (no KD)                       10.16%     2.6388
Distilled student (pruned + KD)               9.38%     2.3724
